# Fixation mRNN Loss Necessity Checks

Train or load a small set of loss ablations, then compare whether each fit reproduces both region-PC time courses and PC-backprojected firing-rate time courses. Keep this notebook unexecuted in git; save generated figures outside the notebook when needed.

## 1. Setup

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())

import sys
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from dal_monte_2022_analysis.ephys.modeling import (
    FixationMRNNRunSettings,
    backproject_replay_outputs_to_firing_rates,
    load_fixation_mrnn_config,
    make_targets,
    pc_reconstructed_firing_rate_accuracy,
    reconstruction_accuracy,
    replay_fixation_mrnn_run,
    resolve_fixation_mrnn_output_root,
    settings_from_config,
    train_fixation_mrnn_scratch,
)

plt.rcParams.update({"figure.dpi": 130})

## 2. Loss Ablation Grid

In [ ]:
cfg = load_fixation_mrnn_config(repo_root / "configs/ephys_fixation_mrnn.yaml")

base_settings = settings_from_config(
    cfg,
    overrides={
        "target_mode": "region_pcs",
        "temporal_basis_count": 0,
        "initialization_mode": "single",
        "device": "auto",
    },
)

loss_variants = {
    "pc_reconstruction_only": dict(
        temporal_derivative_loss_scale=0.0,
        temporal_curvature_loss_scale=0.0,
        correlation_loss_scale=0.0,
        variance_loss_scale=0.0,
        fr_reconstruction_loss_scale=0.0,
        fr_temporal_derivative_loss_scale=0.0,
        fr_temporal_curvature_loss_scale=0.0,
    ),
    "pc_derivatives": dict(
        temporal_derivative_loss_scale=1.0,
        temporal_curvature_loss_scale=1.0,
        correlation_loss_scale=0.0,
        variance_loss_scale=0.0,
        fr_reconstruction_loss_scale=0.0,
        fr_temporal_derivative_loss_scale=0.0,
        fr_temporal_curvature_loss_scale=0.0,
    ),
    "pc_corr_var": dict(
        temporal_derivative_loss_scale=1.0,
        temporal_curvature_loss_scale=1.0,
        correlation_loss_scale=1.0,
        variance_loss_scale=1.0,
        fr_reconstruction_loss_scale=0.0,
        fr_temporal_derivative_loss_scale=0.0,
        fr_temporal_curvature_loss_scale=0.0,
    ),
    "pc_plus_fr": dict(
        temporal_derivative_loss_scale=1.0,
        temporal_curvature_loss_scale=1.0,
        correlation_loss_scale=1.0,
        variance_loss_scale=1.0,
        fr_reconstruction_loss_scale=1.0,
        fr_temporal_derivative_loss_scale=1.0,
        fr_temporal_curvature_loss_scale=1.0,
    ),
}

run_mode = "load"  # use "train" to fit missing variants
overwrite = False
scratch_prefix = "loss_necessity"

## 3. Train or Load Runs

In [ ]:
runs = {}
output_root = resolve_fixation_mrnn_output_root(base_settings) / "scratch"

for label, overrides in loss_variants.items():
    settings = replace(base_settings, **overrides)
    scratch_id = f"{scratch_prefix}_{label}"
    run_dir = output_root / scratch_id
    checkpoint_path = run_dir / "checkpoint_final.pth"
    if run_mode == "train" or not checkpoint_path.exists():
        result = train_fixation_mrnn_scratch(settings, scratch_id=scratch_id, overwrite=overwrite)
        run_dir = Path(result["run_dir"])
    replay = replay_fixation_mrnn_run(run_dir, device=settings.device)
    runs[label] = {"settings": settings, "run_dir": run_dir, "replay": replay}

list(runs)

## 4. Difference Metrics

In [ ]:
metric_tables = []
for label, bundle in runs.items():
    pc_metrics = reconstruction_accuracy(bundle["replay"]).assign(metric_space="region_pcs", loss_variant=label)
    fr_metrics = pc_reconstructed_firing_rate_accuracy(bundle["replay"]).assign(
        metric_space="backprojected_fr",
        loss_variant=label,
    )
    metric_tables.extend([pc_metrics, fr_metrics])

metrics = pd.concat(metric_tables, ignore_index=True)
summary = (
    metrics.groupby(["loss_variant", "metric_space", "region"], as_index=False)
    .agg(mse=("mse", "mean"), mae=("mae", "mean"), r2=("r2", "mean"), correlation=("correlation", "mean"))
    .sort_values(["metric_space", "region", "mse"])
)
summary

## 5. PC Time-Course Overlays

In [ ]:
targets = make_targets(base_settings)
timeline = np.asarray(targets.timeline_s, dtype=float)
region = "ofc"
condition = "face_interactive"
pc_idx = 0

fig, ax = plt.subplots(figsize=(7, 3))
cond_idx = targets.condition_order.index(condition)
ax.plot(timeline, targets.pcs_by_region[region][cond_idx, :, pc_idx], color="black", linewidth=2.0, label="target")
for label, bundle in runs.items():
    replay = bundle["replay"]
    yhat = replay["output_by_region"][region].detach().cpu().numpy()[cond_idx, :, pc_idx]
    ax.plot(timeline, yhat, linewidth=1.2, label=label)
ax.axvline(0.0, color="0.4", linewidth=0.8)
ax.set(title=f"{region} {condition} PC{pc_idx + 1}", xlabel="time (s)", ylabel="score")
ax.legend(frameon=False, fontsize=7)
fig.tight_layout()

## 6. Backprojected Firing-Rate Overlays

In [ ]:
region = "ofc"
condition = "face_interactive"
unit_idx = 0
cond_idx = targets.condition_order.index(condition)
target_fr = targets.pc_reconstructed_raw_by_region()[region][cond_idx, :, unit_idx]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(timeline, target_fr, color="black", linewidth=2.0, label="target PC backprojection")
for label, bundle in runs.items():
    predicted_fr = backproject_replay_outputs_to_firing_rates(bundle["replay"])[region][cond_idx, :, unit_idx]
    ax.plot(timeline, predicted_fr, linewidth=1.2, label=label)
ax.axvline(0.0, color="0.4", linewidth=0.8)
ax.set(title=f"{region} {condition} backprojected unit {unit_idx}", xlabel="time (s)", ylabel="normalized FR")
ax.legend(frameon=False, fontsize=7)
fig.tight_layout()

## 7. Decision Table

In [ ]:
decision_table = (
    summary.groupby(["loss_variant", "metric_space"], as_index=False)
    .agg(mean_mse=("mse", "mean"), mean_mae=("mae", "mean"), mean_r2=("r2", "mean"), mean_corr=("correlation", "mean"))
    .sort_values(["metric_space", "mean_mse"])
)
decision_table